# SHBT Holographic Warp Drive Explorer

Interactive notebook for exploring the SHBT warp metrics, stress-energy audits, and modular register entanglement density.  Move the sliders below to adjust the bubble radius, phase-lock angle, and wall steepness; the Rust/PyO3 core recomputes the full simulation and the plots update live.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
from shbt_warp import Simulation

## Interactive simulation panel

Use the sliders to change the warp-bubble geometry and phase lock.  Larger `grid_points` values give smoother plots but take longer to compute.

In [ ]:
def run_and_plot(radius=10.0, phase=0.421, wall_steepness=0.8, grid_points=301):
    """Run the SHBT simulation and render diagnostic plots."""
    sim = Simulation(
        radius=radius,
        domain_radius=30.0,
        grid_points=int(grid_points),
        wall_steepness=wall_steepness,
        phase=phase,
        stress_grid_points=7,
    )
    result = sim.run()

    display(Markdown(
        f"""
        **Radius**: `{radius:.2f}` m  
        **Phase**: `{phase:.4f}` rad  
        **Wall steepness**: `{wall_steepness:.3f}` 1/m  
        **Effective warp velocity**: `{result['v_eff_c']:.8f} c`  
        **Operational power**: `{result['power_mw']:.4f} MW`  
        **Entropy debt**: `{result['delta_mod']:.6f}`  
        **Boundary audit**: `{"PASS" if result['boundary']["audit"]["passed"] else "FAIL"}`  
        **Stress-energy audit**: `{"PASS" if result['stress_energy']["audit"]["passed"] else "FAIL"}`
        """
    ))

    fig, axes = plt.subplots(2, 2, figsize=(11, 9))

    # 1-D stress-energy audit along the x-axis
    ax = axes[0, 0]
    se = result['stress_energy']
    x = np.array(se['x_m'])
    ax.plot(x, se['ricci_scalar'], 'g-', lw=1.2, label=r'$R$')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('Ricci scalar', color='g')
    ax.tick_params(axis='y', labelcolor='g')
    ax2 = ax.twinx()
    ax2.plot(x, se['energy_density_t00'], 'm--', lw=1.2, label=r'$T_{00}^{\mathrm{eff}}$')
    ax2.set_ylabel(r'Effective $T_{00}$', color='m')
    ax2.tick_params(axis='y', labelcolor='m')
    ax.set_title('Stress-energy audit (center line)')

    # Shift profile
    ax = axes[0, 1]
    fg = result['fg_slice']
    ax.plot(fg['x_m'], fg['beta_over_c'], 'r-', lw=1.2)
    ax.axhline(-fg['v_eff_c'], color='k', ls=':', lw=0.8)
    ax.set_xlabel('x (m)')
    ax.set_ylabel(r'$\beta_x / c$')
    ax.set_title('Shift profile')

    # Baseline Shannon density
    ax = axes[1, 0]
    base = np.array(result['boundary']['shannon_density']).reshape(3, 3)
    im = ax.imshow(base, cmap='viridis', origin='upper', extent=(-0.5, 2.5, 2.5, -0.5))
    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xlabel('SU(3) weight index')
    ax.set_ylabel('SU(2) charge index')
    ax.set_title('Baseline Shannon density')
    fig.colorbar(im, ax=ax, shrink=0.8)

    # Excited Shannon contributions
    ax = axes[1, 1]
    excited = np.array(result['excitation']['excited_shannon_contributions']).reshape(3, 3)
    im = ax.imshow(excited, cmap='plasma', origin='upper', extent=(-0.5, 2.5, 2.5, -0.5))
    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xlabel('SU(3) weight index')
    ax.set_ylabel('SU(2) charge index')
    ax.set_title('Excited Shannon contributions')
    fig.colorbar(im, ax=ax, shrink=0.8)

    plt.tight_layout()
    plt.show()

    return result

In [ ]:
widgets.interact(
    run_and_plot,
    radius=widgets.FloatSlider(min=2.0, max=25.0, step=1.0, value=10.0, description='Radius (m)'),
    phase=widgets.FloatSlider(min=0.0, max=1.5, step=0.05, value=0.421, description='Phase θ'),
    wall_steepness=widgets.FloatSlider(min=0.1, max=2.0, step=0.1, value=0.8, description='Wall steepness'),
    grid_points=widgets.IntSlider(min=101, max=1201, step=100, value=301, description='Grid points'),
)

## Transient and sweep access

The returned result dictionary also contains the time-stepping transient (`result['transient']`) and the de-rendering re-rendering trajectory (`result['derender']['rerender_trajectory']`).  These can be plotted with standard Matplotlib commands.